<a href="https://colab.research.google.com/github/pranavchauhann/sentiment-finetuning/blob/main/sentiment_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Fine-Tuning with DistilBERT

This notebook demonstrates how to fine-tune a pretrained DistilBERT model for binary sentiment classification using the IMDb movie review dataset.

### Goal
Classify movie reviews into:

- `0` = Negative
- `1` = Positive

### What we will learn

- Loading a dataset from Hugging Face
- Train / Validation / Test split
- Tokenization
- DistilBERT
- Padding and attention masks
- Fine-tuning
- Loss and backpropagation
- Accuracy evaluation
- Before vs After model comparison

### Model

`distilbert-base-uncased`

### Dataset

IMDb Movie Reviews Dataset

## Checking GPU Availability

Fine-tuning transformer models is computationally expensive.

Google Colab can provide access to a GPU, which makes training much faster compared to using only the CPU.

The following code checks whether PyTorch can detect and use the GPU.

In [2]:
import torch

print(torch.cuda.is_available())

True


## Checking Required Libraries

This project uses three main Python libraries:

- `PyTorch` → for deep learning and model training
- `Transformers` → for loading and fine-tuning DistilBERT
- `Datasets` → for loading and managing the IMDb dataset

We also print their versions so the environment is easy to reproduce and debug.

In [3]:
import torch
import transformers
import datasets

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

PyTorch: 2.11.0+cu128
Transformers: 5.16.1
Datasets: 4.0.0


## Loading the IMDb Dataset

For this project, we will use the IMDb movie review dataset from the Hugging Face Hub.

Each example contains:

- `text` → the movie review
- `label` → the sentiment class

Label mapping:

- `0` = Negative
- `1` = Positive

We will use the labeled training and test splits for our sentiment classification task.

In [4]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

## Inspecting the Dataset Structure

Before using the dataset, we should understand how it is organized.

The IMDb dataset contains multiple splits such as training, test, and unsupervised data.

We will inspect the dataset to see:
- available splits
- number of rows
- available columns

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

## Inspecting a Single Training Example

Let's inspect one movie review from the training dataset.

Each row contains:

- `text` → the actual movie review
- `label` → the sentiment class

This helps us understand what one training example looks like before preprocessing.

In [6]:
dataset["train"][0]


{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

### Checking a Positive Review

Now we inspect another training example to see a positive sentiment review.

In this dataset:
- `0` = Negative
- `1` = Positive

In [7]:
dataset["train"][20000]

{'text': "After reading some quite negative views for this movie, I was not sure whether I should fork out some money to rent it. However, it was a pleasant surprise. I haven't seen the original movie, but if its better than this, I'd be in heaven.<br /><br />Tom Cruise gives a strong performance as the seemingly unstable David, convincing me that he is more than a smile on legs (for only the third time in his career- the other examples were Magnolia and Born on the Fourth of July). Penelope Cruz is slightly lightweight but fills the demands for her role, as does Diaz. The only disappointment is the slightly bland Kurt Russell. In the movie, however, it is not the acting that really impresses- its the filmmaking.<br /><br />Cameron Crowe excels in the director's role, providing himself with a welcome change of pace from his usual schtick. The increasing insanity of the movie is perfectly executed by Crowe (the brief sequence where Cruise walks through an empty Time Square is incredibly

### Checking Label Distribution

Before training, we should check how many positive and negative examples are present in the training dataset.

A balanced dataset usually helps the model learn both classes fairly.

In [8]:
from collections import Counter

label_counts = Counter(dataset["train"]["label"])
label_counts

Counter({0: 12500, 1: 12500})

### Viewing Text and Label Separately

Each training example contains two main fields:

- `text` → the movie review
- `label` → the sentiment class

We can access them separately to understand the dataset structure more clearly.

In [9]:
example = dataset["train"][0]

print("Review:")
print(example["text"])

print("\nLabel:")
print(example["label"])

Review:
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far betw

### Creating a Validation Set

The IMDb dataset already has training and test sets, but it does not provide a separate validation set.

We will split the training data into:

- 90% Training
- 10% Validation

The model will learn from the training set, while the validation set will help us monitor performance during fine-tuning.

In [10]:
split_dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

split_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 22500
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2500
    })
})

### Renaming the Validation Split

The `train_test_split()` function names the second split as `test` by default.

In our project, this split will be used for validation, so we will store it with a clearer name.

In [11]:
train_dataset = split_dataset["train"]
validation_dataset = split_dataset["test"]
test_dataset = dataset["test"]

print("Train:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Test:", len(test_dataset))

Train: 22500
Validation: 2500
Test: 25000


### Loading the DistilBERT Tokenizer

Transformer models cannot directly understand raw text.

A tokenizer converts text into numerical tokens that the model can process.

We will use the tokenizer that belongs to the pretrained DistilBERT model.

In [12]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

### Tokenizing a Single Sentence

Let's pass a simple sentence through the DistilBERT tokenizer.

The tokenizer will convert the text into numerical IDs that DistilBERT can understand.

In [13]:
sentence = "I love this movie"

encoded = tokenizer(sentence)

encoded

{'input_ids': [101, 1045, 2293, 2023, 3185, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}

### Viewing the Tokens

We can convert the numerical token IDs back into readable tokens.

This helps us understand how the tokenizer split the sentence.

In [14]:
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])

tokens

['[CLS]', 'i', 'love', 'this', 'movie', '[SEP]']

### Understanding Subword Tokenization

DistilBERT does not always treat one word as one token.

If a word is uncommon or complex, the tokenizer may split it into smaller subword pieces.

In [15]:
word = "unbelievable"

tokens = tokenizer.tokenize(word)

tokens

['unbelievable']

### Seeing Subword Tokenization in Action

Some uncommon words are not stored as complete tokens in the tokenizer vocabulary.

In that case, the tokenizer splits them into smaller subword pieces.

In [16]:
word = "unbelievableness"

tokens = tokenizer.tokenize(word)

tokens

['unbelievable', '##ness']

### Creating a Tokenization Function

We will create a small function that takes a batch of movie reviews and converts them into tokenized inputs for DistilBERT.

We will use truncation so that very long reviews do not exceed the model's maximum input length.

In [17]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True
    )

### Testing the Tokenization Function

Before applying the function to the entire dataset, we will test it on a single training example.

This helps us verify that the function is working correctly.

In [18]:
sample = train_dataset[0]

tokenized_sample = tokenize_function(sample)

tokenized_sample

{'input_ids': [101, 2007, 2122, 2111, 6904, 6834, 2061, 2116, 7171, 1010, 2478, 2214, 8333, 1010, 1998, 3806, 7741, 4176, 2000, 2131, 2068, 2041, 1010, 2025, 2000, 5254, 2008, 2070, 1997, 1996, 5019, 2020, 6361, 2006, 1037, 2580, 2275, 2007, 5889, 1010, 2054, 1005, 1055, 2000, 2903, 1029, 2214, 2143, 1997, 3032, 2003, 3835, 1010, 2021, 1996, 4111, 6905, 1998, 16627, 1997, 12493, 2003, 9145, 2000, 3422, 1999, 2122, 3152, 1012, 1045, 2113, 1010, 14398, 2003, 7929, 1999, 2122, 2214, 3152, 1010, 2021, 2045, 2003, 2062, 2000, 2008, 2000, 2191, 2023, 3232, 4558, 21553, 1012, 6791, 2004, 13109, 10136, 1010, 2027, 2196, 5520, 2037, 9738, 1010, 3235, 3779, 2001, 2019, 4654, 1011, 12436, 12672, 26548, 2937, 1010, 2109, 2814, 2066, 2990, 2414, 2005, 3361, 5114, 2096, 10551, 2075, 2068, 1997, 25335, 1010, 16039, 2010, 2564, 1005, 1055, 6835, 6245, 1010, 2478, 2014, 2004, 1037, 10140, 17678, 1010, 2035, 2023, 3084, 2122, 3152, 24257, 1012, 2027, 2020, 2011, 2053, 2965, 1996, 2034, 2000, 3604, 2000,

### Understanding Padding

When multiple sentences are processed together in a batch, they may have different lengths.

Padding adds extra `[PAD]` tokens to shorter sentences so that all inputs in the batch have the same length.

The attention mask uses:
- `1` for real tokens
- `0` for padding tokens

In [19]:
sentences = [
    "I love this movie",
    "This movie was absolutely amazing and I really enjoyed watching it"
]

encoded_batch = tokenizer(
    sentences,
    padding=True
)

encoded_batch

{'input_ids': [[101, 1045, 2293, 2023, 3185, 102, 0, 0, 0, 0, 0, 0, 0], [101, 2023, 3185, 2001, 7078, 6429, 1998, 1045, 2428, 5632, 3666, 2009, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

### Tokenizing the Full Dataset

Now that our tokenization function is working correctly, we will apply it to the training, validation, and test datasets.

We use batched processing so multiple examples are tokenized efficiently.

In [20]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_validation = validation_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/22500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

### Inspecting the Tokenized Dataset

After tokenization, new model-ready fields are added to each example.

Let's inspect the dataset structure to see which columns are now available.

In [21]:
tokenized_train

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 22500
})

### Inspecting One Tokenized Example

Let's inspect one tokenized training example.

This will help us see the original review, its label, and the numerical inputs created for DistilBERT.

In [22]:
sample = tokenized_train[0]

print("Label:", sample["label"])
print("Input IDs:", sample["input_ids"])
print("Attention Mask:", sample["attention_mask"])

Label: 0
Input IDs: [101, 2007, 2122, 2111, 6904, 6834, 2061, 2116, 7171, 1010, 2478, 2214, 8333, 1010, 1998, 3806, 7741, 4176, 2000, 2131, 2068, 2041, 1010, 2025, 2000, 5254, 2008, 2070, 1997, 1996, 5019, 2020, 6361, 2006, 1037, 2580, 2275, 2007, 5889, 1010, 2054, 1005, 1055, 2000, 2903, 1029, 2214, 2143, 1997, 3032, 2003, 3835, 1010, 2021, 1996, 4111, 6905, 1998, 16627, 1997, 12493, 2003, 9145, 2000, 3422, 1999, 2122, 3152, 1012, 1045, 2113, 1010, 14398, 2003, 7929, 1999, 2122, 2214, 3152, 1010, 2021, 2045, 2003, 2062, 2000, 2008, 2000, 2191, 2023, 3232, 4558, 21553, 1012, 6791, 2004, 13109, 10136, 1010, 2027, 2196, 5520, 2037, 9738, 1010, 3235, 3779, 2001, 2019, 4654, 1011, 12436, 12672, 26548, 2937, 1010, 2109, 2814, 2066, 2990, 2414, 2005, 3361, 5114, 2096, 10551, 2075, 2068, 1997, 25335, 1010, 16039, 2010, 2564, 1005, 1055, 6835, 6245, 1010, 2478, 2014, 2004, 1037, 10140, 17678, 1010, 2035, 2023, 3084, 2122, 3152, 24257, 1012, 2027, 2020, 2011, 2053, 2965, 1996, 2034, 2000, 3604,

### Loading DistilBERT for Sentiment Classification

Now we load the pretrained DistilBERT model.

Since our task has two classes:
- 0 = Negative
- 1 = Positive

we configure the model with two output labels.

In [23]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Inspecting the Model Size

DistilBERT contains millions of trainable parameters.

These parameters are the weights that can be adjusted during fine-tuning.

In [24]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

Total parameters: 66955010
Trainable parameters: 66955010


### Testing the Model Before Fine-Tuning

Before training, we will give the model a simple movie review.

The classification head is not yet trained for our sentiment task, so the prediction may be incorrect or uncertain.

Later, we will compare this with the prediction after fine-tuning.

In [25]:
import torch

text = "I absolutely loved this movie. It was fantastic!"

inputs = tokenizer(
    text,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(**inputs)

outputs.logits

tensor([[-0.1520,  0.0779]])

### Converting Logits to Probabilities

The model outputs raw scores called logits.

We can use Softmax to convert these logits into probabilities for each class.

In [26]:
probabilities = torch.softmax(outputs.logits, dim=1)

probabilities

tensor([[0.4428, 0.5572]])

### Getting the Predicted Class

The model assigns a probability to each sentiment class.

We select the class with the highest probability as the final prediction.

In [27]:
predicted_class = torch.argmax(probabilities, dim=1).item()

print("Predicted class:", predicted_class)
print("Predicted sentiment:", "Positive" if predicted_class == 1 else "Negative")

Predicted class: 1
Predicted sentiment: Positive


### Defining Training Arguments

Training arguments control how the fine-tuning process will run.

We will define:
- number of epochs
- batch size
- learning rate
- logging and evaluation behavior

In [28]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100
)

### Setting Up Dynamic Padding

Movie reviews have different lengths.

Instead of padding every review to a fixed maximum length, we will pad dynamically within each batch.

This saves memory and makes training more efficient.

In [29]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### Defining the Accuracy Metric

During validation, we want to measure how many sentiment predictions are correct.

Accuracy is calculated as:

Correct Predictions / Total Predictions

In [30]:
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy
    }

### Setting Up the Trainer

The Trainer connects our model, datasets, training settings, padding logic, and evaluation metric.

It will manage the fine-tuning loop for us.

In [31]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

### Starting Fine-Tuning

Now we will start training the model on the IMDb sentiment dataset.

During training, the model will:

1. Process a batch of reviews
2. Make predictions
3. Calculate loss
4. Backpropagate the error
5. Update the weights
6. Repeat for all batches and epochs

In [32]:
trainer.train()

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy
1,0.225617,0.233622,0.910800
2,0.149502,0.260601,0.922000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2814, training_loss=0.2071774003771746, metrics={'train_runtime': 2268.4697, 'train_samples_per_second': 19.837, 'train_steps_per_second': 1.24, 'total_flos': 5896822472726160.0, 'train_loss': 0.2071774003771746, 'epoch': 2.0})

## Training Results

The model was fine-tuned for 2 epochs.

### Epoch 1
- Training Loss: 0.2256
- Validation Loss: 0.2336
- Validation Accuracy: 91.08%

### Epoch 2
- Training Loss: 0.1495
- Validation Loss: 0.2606
- Validation Accuracy: 92.20%

The training loss decreased, which shows that the model learned from the training data.

Validation accuracy improved to 92.20%.

However, validation loss increased slightly in the second epoch, which may indicate the beginning of overfitting.

## Evaluating on the Test Set

Now we evaluate the fine-tuned model on the test dataset.

The test set was not used during training or validation, so it gives us a more realistic estimate of how well the model performs on unseen data.

In [34]:
test_results = trainer.evaluate(tokenized_test)

test_results

Training Loss,Validation Loss,Epoch,Accuracy
0.149502,0.223158,2,0.931720


{'eval_loss': 0.22315798699855804, 'eval_accuracy': 0.93172}

## Test Set Results

The fine-tuned model was evaluated on the unseen IMDb test dataset.

### Results
- Test Loss: 0.2232
- Test Accuracy: 93.17%

The model performs well on unseen data, which indicates that the fine-tuning process was successful.

## Comparing Before and After Fine-Tuning

We will test the same sentence again after fine-tuning.

Before fine-tuning, the model was only slightly leaning toward the positive class.

Now we will check whether the fine-tuned model gives a stronger and more confident sentiment prediction.

In [36]:
text = "I absolutely loved this movie. It was fantastic!"

inputs = tokenizer(
    text,
    return_tensors="pt"
)

inputs = {key: value.to(model.device) for key, value in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=1)

predicted_class = torch.argmax(probabilities, dim=1).item()

print("Negative probability:", probabilities[0][0].item())
print("Positive probability:", probabilities[0][1].item())
print("Predicted class:", predicted_class)
print("Predicted sentiment:", "Positive" if predicted_class == 1 else "Negative")

Negative probability: 0.004740655422210693
Positive probability: 0.9952593445777893
Predicted class: 1
Predicted sentiment: Positive


## Before vs After Fine-Tuning

We tested the same positive sentence before and after fine-tuning.

### Before Fine-Tuning
- Negative Probability: 46.52%
- Positive Probability: 53.48%
- Prediction: Positive

The model was almost uncertain between the two classes.

### After Fine-Tuning
- Negative Probability: 0.47%
- Positive Probability: 99.53%
- Prediction: Positive

After fine-tuning, the model became much more confident in identifying positive sentiment.

## Saving the Fine-Tuned Model

After fine-tuning, we save both the trained model and tokenizer.

This allows us to load the model later and make predictions without training it again.

In [37]:
save_path = "./sentiment-distilbert"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


## Reloading the Saved Model

To verify that the model was saved correctly, we will load it again from the saved folder.

If it loads successfully, we can reuse the fine-tuned model later without retraining.

In [38]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

loaded_model = AutoModelForSequenceClassification.from_pretrained(
    "./sentiment-distilbert"
)

loaded_tokenizer = AutoTokenizer.from_pretrained(
    "./sentiment-distilbert"
)

print("Model and tokenizer loaded successfully!")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model and tokenizer loaded successfully!


## Verifying the Reloaded Model

We will make a prediction using the reloaded model.

If the prediction is similar to the fine-tuned model's earlier output, it confirms that the model was saved and loaded correctly.

In [39]:
text = "I absolutely loved this movie. It was fantastic!"

inputs = loaded_tokenizer(
    text,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = loaded_model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=1)

predicted_class = torch.argmax(probabilities, dim=1).item()

print("Negative probability:", probabilities[0][0].item())
print("Positive probability:", probabilities[0][1].item())
print("Predicted sentiment:", "Positive" if predicted_class == 1 else "Negative")

Negative probability: 0.00474065076559782
Positive probability: 0.9952593445777893
Predicted sentiment: Positive


## Reload Verification

The saved fine-tuned model and tokenizer were loaded successfully.

We tested the same positive sentence again and received nearly the same prediction:

- Negative Probability: 0.47%
- Positive Probability: 99.53%
- Prediction: Positive

This confirms that the trained model was saved and restored correctly.

## Uploading the Fine-Tuned Model to Hugging Face Hub

We will upload the trained model and tokenizer to the Hugging Face Hub.

This allows us to:
- store the model permanently
- load it later without retraining
- share the model using a repository name

In [40]:
from huggingface_hub import notebook_login

notebook_login()

## Pushing the Model to Hugging Face Hub

Now that we are authenticated, we will upload the fine-tuned model and tokenizer to Hugging Face Hub.

This will create a reusable model repository that we can load later directly from Hugging Face.

In [41]:
repo_name = "sentiment-distilbert-imdb"

model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...u0_n3vw/model.safetensors:   2%|1         | 4.67MB /  268MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/pranavchauhann/sentiment-distilbert-imdb/commit/9ffe7eac9a5b0bd3a897339f69f03d5179c6984c', commit_message='Upload tokenizer', commit_description='', oid='9ffe7eac9a5b0bd3a897339f69f03d5179c6984c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/pranavchauhann/sentiment-distilbert-imdb', endpoint='https://huggingface.co', repo_type='model', repo_id='pranavchauhann/sentiment-distilbert-imdb'), pr_revision=None, pr_num=None)

## Loading the Model from Hugging Face Hub

We will now load the fine-tuned model directly from Hugging Face Hub.

If this works, it confirms that the model is permanently stored online and can be reused without retraining.

In [44]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

hub_model = AutoModelForSequenceClassification.from_pretrained(
    "pranavchauhann/sentiment-distilbert-imdb"
)

hub_tokenizer = AutoTokenizer.from_pretrained(
    "pranavchauhann/sentiment-distilbert-imdb"
)

print("Model loaded successfully from Hugging Face Hub!")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

Model loaded successfully from Hugging Face Hub!


## Verifying the Hugging Face Hub Model

Now we will make a prediction using the model loaded directly from Hugging Face Hub.

If the prediction matches our earlier fine-tuned result, it confirms that the uploaded model is working correctly.

In [45]:
text = "I absolutely loved this movie. It was fantastic!"

inputs = hub_tokenizer(
    text,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = hub_model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=1)

predicted_class = torch.argmax(probabilities, dim=1).item()

print("Negative probability:", probabilities[0][0].item())
print("Positive probability:", probabilities[0][1].item())
print("Predicted sentiment:", "Positive" if predicted_class == 1 else "Negative")

Negative probability: 0.00474065076559782
Positive probability: 0.9952593445777893
Predicted sentiment: Positive


## Final Summary

In this project, we fine-tuned DistilBERT for IMDb sentiment classification.

### What we learned

- How pretrained models work
- How tokenization converts text into model inputs
- Input IDs, attention masks, and padding
- Train / Validation / Test splitting
- Epochs, batch size, and learning rate
- Fine-tuning with Hugging Face Trainer
- Loss, backpropagation, and weight updates
- Model evaluation using accuracy
- Saving and reloading a trained model
- Uploading a fine-tuned model to Hugging Face Hub

### Final Test Accuracy

**93.17%**